# Complete IFBC Buddhist Books Downloader

**Based on diagnostic findings:**
- The site has 156 posts total, mostly organized by author/teacher
- Only 2 Tripitaka-specific categories found
- We'll download from ALL relevant categories to ensure we get everything

**Strategy:**
1. Download from both Tripitaka categories
2. Check if individual posts contain multiple PDFs (likely!)
3. Organize by post/section rather than by Tripitaka division

In [1]:
!pip install -q requests beautifulsoup4 gdown tqdm lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import gdown
import os
import re
from urllib.parse import urlparse
from tqdm.auto import tqdm
import json
from datetime import datetime
import time

In [3]:
# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

# Configuration
PROJECT_DIR = '/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/'
TRIPITAKA_DIR = os.path.join(PROJECT_DIR, 'data/raw/tripitaka')
os.makedirs(TRIPITAKA_DIR, exist_ok=True)

HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}

# Target categories
CATEGORIES = [
    'https://download.ifbcnet.org/category/thripitaka-books/',
    'https://download.ifbcnet.org/category/books-related-to-the-tipitaka/',
]

print(f'📁 Output directory: {TRIPITAKA_DIR}')

Mounted at /content/drive
📁 Output directory: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka


## Helper Functions

In [4]:
def clean_name(text):
    """Clean text for use as folder/file name."""
    text = re.sub(r'[<>:"/\\|?*]', '_', text)
    text = re.sub(r'\s+', '_', text)
    text = text.strip('._')
    return text[:100]  # Limit length

def extract_google_drive_id(url):
    """Extract file ID from Google Drive URL."""
    patterns = [
        r'drive\.google\.com/file/d/([a-zA-Z0-9_-]+)',
        r'drive\.google\.com/open\?id=([a-zA-Z0-9_-]+)',
        r'/d/([a-zA-Z0-9_-]+)',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None

def download_from_gdrive(file_id, output_path, max_retries=3):
    """Download file from Google Drive with retries."""
    for attempt in range(max_retries):
        try:
            url = f'https://drive.google.com/uc?id={file_id}'
            gdown.download(url, output_path, quiet=False)
            if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                return True
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return False

## Comprehensive Post Discovery

In [5]:
def get_all_posts_from_sitemap():
    """Get ALL posts from sitemap, then filter for Tripitaka-related."""
    print('='*80)
    print('FETCHING ALL POSTS FROM SITEMAP')
    print('='*80)

    sitemap_url = 'https://download.ifbcnet.org/post-sitemap.xml'

    try:
        resp = requests.get(sitemap_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'lxml-xml')

        all_urls = [loc.text for loc in soup.find_all('loc')]
        print(f'Total posts in sitemap: {len(all_urls)}')

        # Filter for Tripitaka-related posts
        # We'll be more permissive and check each post's content
        tripitaka_keywords = ['tripitaka', 'thripitaka', 'tipitaka', 'ත්‍රිපිටක']

        relevant_posts = []

        print('\nChecking posts for Tripitaka-related content...')
        for url in tqdm(all_urls[:50], desc='Scanning posts'):  # Check first 50 to start
            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                if resp.status_code == 200:
                    text = resp.text.lower()

                    # Check if post contains Tripitaka keywords
                    if any(kw in text for kw in tripitaka_keywords):
                        # Get title
                        soup = BeautifulSoup(resp.content, 'html.parser')
                        title_tag = soup.find('h1') or soup.find('title')
                        title = title_tag.get_text(strip=True) if title_tag else 'Unknown'

                        relevant_posts.append({
                            'url': url,
                            'title': title
                        })

                time.sleep(0.3)
            except:
                continue

        print(f'\n✓ Found {len(relevant_posts)} Tripitaka-related posts')
        return relevant_posts

    except Exception as e:
        print(f'Error fetching sitemap: {e}')
        return []

In [6]:
def get_posts_from_categories(category_urls, max_pages=10):
    """Get all posts from specified categories."""
    print('='*80)
    print('SCANNING CATEGORY PAGES')
    print('='*80)

    all_posts = []
    seen = set()

    for cat_url in category_urls:
        print(f'\nCategory: {cat_url}')

        for page in range(1, max_pages + 1):
            url = cat_url if page == 1 else f'{cat_url}page/{page}/'

            try:
                resp = requests.get(url, headers=HEADERS, timeout=30)
                if resp.status_code == 404:
                    break
                resp.raise_for_status()

                soup = BeautifulSoup(resp.content, 'html.parser')
                articles = soup.find_all('article')

                page_count = 0
                for article in articles:
                    # Find title and link
                    title_tag = article.find(['h1', 'h2', 'h3'])
                    if title_tag:
                        link_tag = title_tag.find('a', href=True)
                        if link_tag:
                            post_url = link_tag['href']
                            if post_url not in seen:
                                all_posts.append({
                                    'url': post_url,
                                    'title': title_tag.get_text(strip=True),
                                    'category': cat_url
                                })
                                seen.add(post_url)
                                page_count += 1

                print(f'  Page {page}: {page_count} posts')

                if page_count == 0:
                    break

                time.sleep(1)

            except Exception as e:
                print(f'  Error on page {page}: {e}')
                break

    print(f'\n✓ Total posts found: {len(all_posts)}')
    return all_posts

## PDF Extraction and Download

In [7]:
def get_pdfs_from_post(post_url):
    """Extract ALL PDF links from a post."""
    try:
        resp = requests.get(post_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')

        pdfs = []

        # Find all Google Drive links
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'drive.google.com' in href:
                file_id = extract_google_drive_id(href)
                if file_id:
                    # Try to get a descriptive name from link text or surrounding context
                    link_text = link.get_text(strip=True)

                    # If no text, look for nearby text
                    if not link_text or len(link_text) < 3:
                        parent = link.parent
                        if parent:
                            link_text = parent.get_text(strip=True)[:100]

                    pdfs.append({
                        'file_id': file_id,
                        'url': href,
                        'description': link_text or f'file_{file_id}'
                    })

        time.sleep(0.5)
        return pdfs
    except Exception as e:
        print(f'Error fetching PDFs: {e}')
        return []

In [8]:
def download_post_pdfs(post_info, pdfs):
    """Download all PDFs from a post."""
    post_title = clean_name(post_info['title'])
    post_dir = os.path.join(TRIPITAKA_DIR, post_title)
    os.makedirs(post_dir, exist_ok=True)

    stats = {
        'post': post_title,
        'url': post_info['url'],
        'total': len(pdfs),
        'successful': 0,
        'failed': 0,
        'skipped': 0,
        'files': []
    }

    for idx, pdf in enumerate(tqdm(pdfs, desc=f'{post_title[:40]}'), 1):
        # Generate filename
        desc = clean_name(pdf['description'])
        if not desc or desc == f'file_{pdf["file_id"]}':
            filename = f'{post_title}_{idx}.pdf'
        else:
            filename = f'{desc}.pdf' if not desc.endswith('.pdf') else desc

        filepath = os.path.join(post_dir, filename)

        # Skip if exists
        if os.path.exists(filepath):
            stats['skipped'] += 1
            stats['files'].append({'name': filename, 'status': 'skipped'})
            continue

        # Download
        if download_from_gdrive(pdf['file_id'], filepath):
            stats['successful'] += 1
            stats['files'].append({
                'name': filename,
                'status': 'success',
                'size': os.path.getsize(filepath)
            })
        else:
            stats['failed'] += 1
            stats['files'].append({'name': filename, 'status': 'failed'})

        time.sleep(1)

    return stats

## Main Execution

In [9]:
def main():
    """Complete download pipeline."""
    start = datetime.now()
    print(f'\n🚀 Starting at {start.strftime("%H:%M:%S")}\n')

    # Get all posts from categories
    posts = get_posts_from_categories(CATEGORIES, max_pages=20)

    if not posts:
        print('\n❌ No posts found!')
        return

    # Save post list
    with open(os.path.join(TRIPITAKA_DIR, '_posts.json'), 'w', encoding='utf-8') as f:
        json.dump(posts, f, indent=2, ensure_ascii=False)

    print(f'\n{'='*80}')
    print('DOWNLOADING PDFs FROM ALL POSTS')
    print('='*80)

    results = []

    for i, post in enumerate(posts, 1):
        print(f'\n[{i}/{len(posts)}] {post["title"]}')
        print(f'URL: {post["url"]}')

        # Get PDFs from this post
        pdfs = get_pdfs_from_post(post['url'])
        print(f'PDFs found: {len(pdfs)}')

        if pdfs:
            result = download_post_pdfs(post, pdfs)
            results.append(result)
        else:
            print('⚠️  No PDFs in this post')

        # Save progress
        with open(os.path.join(TRIPITAKA_DIR, '_progress.json'), 'w') as f:
            json.dump(results, f, indent=2)

    # Final summary
    print(f'\n{'='*80}')
    print('FINAL SUMMARY')
    print('='*80)

    total_pdfs = sum(r['total'] for r in results)
    success = sum(r['successful'] for r in results)
    failed = sum(r['failed'] for r in results)
    skipped = sum(r['skipped'] for r in results)

    print(f'Posts processed: {len(results)}')
    print(f'Total PDFs: {total_pdfs}')
    print(f'✓ Downloaded: {success}')
    print(f'⊝ Skipped: {skipped}')
    print(f'✗ Failed: {failed}')

    end = datetime.now()
    print(f'\n⏱️  Duration: {end - start}')

    # Save report
    report = {
        'timestamp': end.isoformat(),
        'duration': str(end - start),
        'summary': {
            'posts': len(results),
            'total_pdfs': total_pdfs,
            'success': success,
            'failed': failed,
            'skipped': skipped
        },
        'details': results
    }

    with open(os.path.join(TRIPITAKA_DIR, '_final_report.json'), 'w') as f:
        json.dump(report, f, indent=2)

    print(f'\n✅ Complete! Report saved to: {TRIPITAKA_DIR}/_final_report.json')

In [ ]:
# RUN THE COMPLETE DOWNLOADER
main()


🚀 Starting at 09:35:59

SCANNING CATEGORY PAGES

Category: https://download.ifbcnet.org/category/thripitaka-books/
  Page 1: 10 posts
  Page 2: 10 posts

Category: https://download.ifbcnet.org/category/books-related-to-the-tipitaka/
  Page 1: 9 posts

✓ Total posts found: 29

DOWNLOADING PDFs FROM ALL POSTS

[1/29] ත්‍රිපිටකය – දීඝනිකාය ( thripitaka – Digha Nikaya)
URL: https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%af%e0%b7%93%e0%b6%9d%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thripitaka-digha-nikaya/
PDFs found: 3


ත්‍රිපිටකය_–_දීඝනිකාය_(_thripitaka_–_Dig:   0%|          | 0/3 [00:00<?, ?it/s]

Downloading...
From (original): https://drive.google.com/uc?id=0B5RD8Fve5lhmeTNhUU8zQ0Z1LUE
From (redirected): https://drive.google.com/uc?id=0B5RD8Fve5lhmeTNhUU8zQ0Z1LUE&confirm=t&uuid=864a92d8-d4df-4b6e-ad81-23e0c6800e6c
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/ත්‍රිපිටකය_–_දීඝනිකාය_(_thripitaka_–_Digha_Nikaya)/දීඝ_නිකාය_1.pdf

  0%|          | 0.00/140M [00:00<?, ?B/s]
  1%|▏         | 2.10M/140M [00:00<00:06, 20.8MB/s]
 13%|█▎        | 18.4M/140M [00:00<00:02, 55.9MB/s]
 21%|██        | 29.4M/140M [00:00<00:02, 55.2MB/s]
 32%|███▏      | 45.1M/140M [00:00<00:01, 79.5MB/s]
 40%|███▉      | 55.6M/140M [00:00<00:01, 79.5MB/s]
 52%|█████▏    | 72.9M/140M [00:00<00:00, 103MB/s] 
 61%|██████    | 84.9M/140M [00:01<00:00, 101MB/s]
 68%|██████▊   | 95.9M/140M [00:01<00:00, 97.7MB/s]
 76%|███████▌  | 106M/140M [00:01<00:00, 71.8MB/s] 
 83%|████████▎ | 116M/140M [00:01<00:00, 76.4MB/s]
100%|██████████| 140M/140M [00:01<00:00, 84.


[2/29] ත්‍රිපිටකය – මජ්ජිම නිකාය ( thripitaka – Majjhima Nikay)
URL: https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%b8%e0%b6%a2%e0%b7%8a%e0%b6%a2%e0%b7%92%e0%b6%b8-%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thri/
PDFs found: 3


ත්‍රිපිටකය_–_මජ්ජිම_නිකාය_(_thripitaka_–:   0%|          | 0/3 [00:00<?, ?it/s]

Downloading...
From (original): https://drive.google.com/uc?id=0B5RD8Fve5lhmc0VUMkFqX1V0dzA
From (redirected): https://drive.google.com/uc?id=0B5RD8Fve5lhmc0VUMkFqX1V0dzA&confirm=t&uuid=a9ef49a6-f070-47ca-9286-0785b519af62
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/ත්‍රිපිටකය_–_මජ්ජිම_නිකාය_(_thripitaka_–_Majjhima_Nikay)/මජ්ක්ධිම_නිකාය_1.pdf

  0%|          | 0.00/177M [00:00<?, ?B/s]
  1%|          | 2.10M/177M [00:00<00:08, 19.6MB/s]
  5%|▌         | 9.44M/177M [00:00<00:07, 23.7MB/s]
  7%|▋         | 13.1M/177M [00:00<00:07, 22.9MB/s]
  9%|▉         | 15.7M/177M [00:00<00:06, 23.1MB/s]
 11%|█         | 18.9M/177M [00:00<00:06, 24.9MB/s]
 12%|█▏        | 21.5M/177M [00:00<00:07, 20.6MB/s]
 15%|█▍        | 26.2M/177M [00:01<00:05, 25.9MB/s]
 17%|█▋        | 29.4M/177M [00:01<00:06, 22.3MB/s]
 18%|█▊        | 32.0M/177M [00:01<00:06, 20.9MB/s]
 22%|██▏       | 38.3M/177M [00:01<00:06, 21.9MB/s]
 25%|██▌       | 44.6M/177M [00


[3/29] ත්‍රිපිටකය – සංයුක්ත නිකාය ( thripitaka – Samyutta Nikaya)
URL: https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b7%83%e0%b6%82%e0%b6%ba%e0%b7%94%e0%b6%9a%e0%b7%8a%e0%b6%ad-%e0%b6%b1%e0%b7%92%e0%b6%9a/
PDFs found: 6


ත්‍රිපිටකය_–_සංයුක්ත_නිකාය_(_thripitaka_:   0%|          | 0/6 [00:00<?, ?it/s]

Downloading...
From: https://drive.google.com/uc?id=0B5RD8Fve5lhmc2tGNmtsdWQ5dzA
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/ත්‍රිපිටකය_–_සංයුක්ත_නිකාය_(_thripitaka_–_Samyutta_Nikaya)/සගාථ_වග්ග.pdf

  0%|          | 0.00/88.5M [00:00<?, ?B/s]
  3%|▎         | 2.62M/88.5M [00:00<00:03, 24.8MB/s]
 15%|█▍        | 13.1M/88.5M [00:00<00:01, 67.4MB/s]
 23%|██▎       | 19.9M/88.5M [00:00<00:01, 66.1MB/s]
 31%|███▏      | 27.8M/88.5M [00:00<00:00, 70.2MB/s]
 40%|███▉      | 35.1M/88.5M [00:00<00:00, 68.1MB/s]
 52%|█████▏    | 46.1M/88.5M [00:00<00:00, 70.6MB/s]
 60%|██████    | 53.5M/88.5M [00:00<00:00, 70.4MB/s]
 69%|██████▊   | 60.8M/88.5M [00:00<00:00, 64.8MB/s]
100%|██████████| 88.5M/88.5M [00:01<00:00, 80.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B5RD8Fve5lhmcTdKc1pfZzlGa2c
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/ත්‍රිපිටකය_–_සංයුක්ත_නිකාය_(_thripi

## Analysis

In [ ]:
# Analyze downloaded content
import glob

sections = [d for d in os.listdir(TRIPITAKA_DIR)
            if os.path.isdir(os.path.join(TRIPITAKA_DIR, d)) and not d.startswith('_')]

print('\nDownloaded Posts:')
print('='*70)

total_files = 0
total_size = 0

for sec in sorted(sections):
    path = os.path.join(TRIPITAKA_DIR, sec)
    pdfs = glob.glob(os.path.join(path, '*.pdf'))
    size = sum(os.path.getsize(p) for p in pdfs)

    total_files += len(pdfs)
    total_size += size

    print(f'{sec[:60]}:')
    print(f'  Files: {len(pdfs):3d} | Size: {size/(1024*1024):6.1f} MB')

print('='*70)
print(f'Total: {total_files} files | {total_size/(1024*1024):.1f} MB')